# Gestión de Memorias: Copias Profundas vs. Superficiales

## 🎯 Objetivos
Uno de los errores más comunes y frustrantes en pandas es modificar accidentalmente el dataset original mientras se realizaban pruebas en una "copia". Esto sucede porque Python maneja los objetos mediante **referencias**.

En este notebook aprenderás a:
1. Diferenciar entre una asignación simple, una copia superficial (*shallow copy*) y una copia profunda (*deep copy*).
2. Utilizar el método `.copy()` para aislar tus experimentos del dataset original.
3. Evitar el error `SettingWithCopyWarning` comprendiendo cómo pandas gestiona la memoria.

## 💡 Introducción

En Python, cuando escribes `df2 = df`, no estás creando un nuevo DataFrame. Estás creando una **nueva etiqueta** que apunta al mismo lugar de la memoria. Si cambias un valor en `df2`, también cambiará en `df` porque ambos son, en realidad, el mismo objeto.

Para evitar esto, pandas nos proporciona el método `.copy()`, que nos permite decidir qué tan "independiente" queremos que sea nuestra copia.

In [ ]:
import pandas as pd
from pathlib import Path

# Configuración del dataset
file_path = Path('players_20.csv')
df = pd.read_csv(file_path)
df.set_index('short_name', inplace=True)
df = df[['long_name', 'age', 'dob', 'height_cm', 'weight_kg', 'nationality', 'club']]

df.head()

## 🛠️ Tipos de "Copiado" en Pandas

### 🌉 Puente Pedagógico: El Mapa de Memoria
Imagina que tu DataFrame es una casa física en una dirección específica.

1. **Asignación (`df2 = df`)**: Es como darle la dirección de la casa a un amigo. Ahora hay dos personas que pueden entrar y cambiar los muebles de la **misma casa**.
2. **Copia Superficial (`deep=False`)**: Es como construir una casa idéntica, pero compartiendo los mismos muebles. Si alguien cambia un mueble, cambia para ambos.
3. **Copia Profunda (`deep=True`)**: Es como construir una casa idéntica con muebles nuevos y propios. Lo que pase en una casa no afecta la otra.

```
  [ OBJETO ORIGINAL ] <--- df
          ^
          | (Referencia)
          |
       df_asignado
```

### 1. Asignación Simple (Solo Referencia)

No crea ninguna copia. Es simplemente un alias.

In [ ]:
# Simple asignación
df_alias = df

# Modificamos el alias
df_alias.loc['L. Messi', 'height_cm'] = 999

print("Valor en df_alias:", df_alias.loc['L. Messi', 'height_cm'])
print("Valor en df original:", df.loc['L. Messi', 'height_cm'])
print("¿Son el mismo objeto?", df is df_alias)

### 2. Copia Profunda (`deep=True`)

Es la opción por defecto de `.copy()`. Crea un objeto totalmente independiente.

In [ ]:
# Resetear el valor previo
df.loc['L. Messi', 'height_cm'] = 170

# Crear copia profunda
df_deep = df.copy(deep=True)

# Modificar la copia
df_deep.loc['L. Messi', 'height_cm'] = 200

print("Valor en df_deep:", df_deep.loc['L. Messi', 'height_cm'])
print("Valor en df original:", df.loc['L. Messi', 'height_cm'])
print("¿Son el mismo objeto?", df is df_deep)

### 3. Copia Superficial (`deep=False`)

Crea un nuevo objeto de DataFrame, pero no copia los datos subyacentes. Es más rápida y consume menos memoria, pero es peligrosa si modificas los datos.

In [ ]:
# Crear copia superficial
df_shallow = df.copy(deep=False)

# Modificar la copia superficial
df_shallow.loc['Cristiano Ronaldo', 'height_cm'] = 250

print("Valor en df_shallow:", df_shallow.loc['Cristiano Ronaldo', 'height_cm'])
print("Valor en df original:", df.loc['Cristiano Ronaldo', 'height_cm'])
print("¿Tienen la misma dirección de memoria?", df is df_shallow)

## 📝 Ejercicios de Práctica

1. **El Experimento Seguro**: Crea una copia profunda de tu DataFrame llamada `df_test`. Cambia la nacionalidad de los primeros 5 jugadores en `df_test` y verifica que el DataFrame `df` original permanezca intacto.
2. **La Trampa de la Referencia**: Crea una asignación simple `df_ref = df`. Intenta cambiar la edad de un jugador en `df_ref` y observa qué sucede con `df`.
3. **Análisis de Memoria**: Investiga qué sucede si creas una copia superficial y luego cambias el nombre de una columna en la copia. ¿Se refleja el cambio de nombre en el original? (Pista: el nombre de la columna es parte de la estructura, no solo de la data).

## 📋 Resumen Rápido

| Método | Tipo de Copia | Independencia de Datos | Uso Recomendado |
| :--- | :--- | :--- | :--- |
| `df2 = df` | **Asignación** | Nula (Mismo objeto) | Solo para alias temporales. |
| `df.copy(deep=False)` | **Superficial** | Parcial (Comparte data) | Cuando solo necesitas cambiar la estructura sin tocar la data. |
| `df.copy(deep=True)` | **Profunda** | Total (Objeto nuevo) | **Siempre** que quieras experimentar sin romper el original. |